# Task 20 - Tuning logistic regression

**Owner:** Viviana Pajic. Implementation and initial explanations were prepared with AI assistance; see [the assistance record](../docs/AI_USE.md).

Logistic regression is our simple, interpretable model. We tune its
regularization strength `C` (small C = simpler model) and `class_weight`
(whether poisonous mushrooms get extra weight). We use the same development
rows, folds and metric as the baselines in notebook 02, so the result is
directly comparable.

## Method: nested cross-validation

If we tried every setting on the same folds we report, the best score would be
too optimistic, because we would pick the setting that happened to fit those
folds. So the search runs inside each of the 10 outer folds: a grid search with
its own 3 inner folds chooses the settings, using only the outer training rows,
and the outer validation fold then scores that choice. The reserved test rows
are never used.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'src/mushrooms.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from mushroom_workflow import REPORTS, candidates, evaluate_model, load_development

X, y, _ = load_development()
baseline = pd.read_csv(REPORTS / 'baseline_cv.csv').set_index('model')
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print('Development rows:', len(X), '| baseline LR AP: %.4f' % baseline.loc['logistic_regression', 'cv_average_precision'])

Development rows: 4000 | baseline LR AP: 0.6307


In [2]:
grid = {'model__C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        'model__class_weight': [None, 'balanced']}
search = GridSearchCV(candidates(X)['logistic_regression'], grid,
                      scoring='average_precision', cv=inner_cv, n_jobs=1)
summary, folds, _ = evaluate_model('logistic_regression_tuned', search, X, y)
pd.DataFrame([summary]).to_csv(REPORTS / 'tuned_logistic_regression_cv.csv', index=False)
folds.to_csv(REPORTS / 'tuned_logistic_regression_folds.csv', index=False)
comparison = pd.DataFrame([baseline.loc['logistic_regression'], pd.Series(summary)],
                          index=['baseline', 'tuned'])
display(comparison[['cv_average_precision', 'cv_average_precision_std',
                    'cv_roc_auc', 'train_average_precision']].astype(float).round(4))

,cv_average_precision,cv_average_precision_std,cv_roc_auc,train_average_precision
baseline,0.6307,0.0103,0.6894,0.6538
tuned,0.6306,0.0105,0.6896,0.6538


## Which settings does the search prefer?

To see what the search actually chooses, we run it once on all 4,000
development rows and list the best settings.

In [3]:
search.fit(X, y)
results = pd.DataFrame(search.cv_results_)[['param_model__C', 'param_model__class_weight',
                                           'mean_test_score', 'std_test_score', 'rank_test_score']]
display(results.sort_values('rank_test_score').head(6).round(4))
print('Chosen settings:', search.best_params_)

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,rank_test_score
12,10,None,0.6254,0.0025,1
8,1,None,0.6253,0.0044,2
10,3,None,0.6253,0.0030,3
11,3,balanced,0.6242,0.0038,4
13,10,balanced,0.6241,0.0035,5
9,1,balanced,0.6241,0.0047,6


Chosen settings: {'model__C': 10, 'model__class_weight': None}


## What this completes

Tuning does not improve logistic regression: the tuned average precision is
practically the same as the baseline, and the best settings differ by less
than their fold-to-fold spread. The reason is visible in the training score,
which is barely higher than the validation score. The model is not overfitting
but underfitting: a linear model cannot combine the weak features the way trees
can, and no value of `C` changes that. We keep this negative result as evidence
that the limit is the model family, not its settings.
The top settings differ by about 0.001 average precision, less than their
standard deviation, and `class_weight='balanced'` never helps: it shifts the
probabilities but does not improve how the model ranks mushrooms, which is what
average precision measures.